In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\navon\OneDrive\Desktop\CrossRank")
data = pd.read_csv(ROOT / "data/processed/uk_etf_features_targets.csv", parse_dates=["date"])

print(f"Rows: {len(data):,}")
print(f"Tickers: {data['ticker'].nunique()}")
print(f"Date range: {data['date'].min().date()} to {data['date'].max().date()}")

# walk-forward config — matches the master prompt's original spec
TRAIN_MONTHS = 36
TEST_MONTHS = 3
FEATURE_COLS = ["mom_1m","mom_3m","mom_6m","mom_12m","realized_vol","max_drawdown_12m"]
TARGET_COL = "xs_excess_ret"

all_dates = sorted(data["date"].unique())
print(f"\nTotal unique month-end dates: {len(all_dates)}")
print(f"First date: {pd.Timestamp(all_dates[0]).date()}  Last date: {pd.Timestamp(all_dates[-1]).date()}")

# how many months do we actually have for walk-forward folds?
n_folds_possible = (len(all_dates) - TRAIN_MONTHS) // TEST_MONTHS
print(f"Approx. walk-forward folds possible: {n_folds_possible}")

Rows: 40,092
Tickers: 825
Date range: 2010-03-31 to 2026-10-31

Total unique month-end dates: 200
First date: 2010-03-31  Last date: 2026-10-31
Approx. walk-forward folds possible: 54


In [2]:
# build the walk-forward fold schedule: (train_start, train_end, test_start, test_end)
folds = []
i = TRAIN_MONTHS
while i + TEST_MONTHS <= len(all_dates):
    train_dates = all_dates[i - TRAIN_MONTHS : i]
    test_dates = all_dates[i : i + TEST_MONTHS]
    folds.append({
        "train_start": train_dates[0], "train_end": train_dates[-1],
        "test_start": test_dates[0], "test_end": test_dates[-1],
    })
    i += TEST_MONTHS

print(f"Total folds: {len(folds)}")
print("\nFirst fold:")
print(folds[0])
print("\nLast fold:")
print(folds[-1])

Total folds: 54

First fold:
{'train_start': Timestamp('2010-03-31 00:00:00'), 'train_end': Timestamp('2013-02-28 00:00:00'), 'test_start': Timestamp('2013-03-31 00:00:00'), 'test_end': Timestamp('2013-05-31 00:00:00')}

Last fold:
{'train_start': Timestamp('2023-06-30 00:00:00'), 'train_end': Timestamp('2026-05-31 00:00:00'), 'test_start': Timestamp('2026-06-30 00:00:00'), 'test_end': Timestamp('2026-08-31 00:00:00')}


In [3]:
try:
    import lightgbm as lgb
    print(f"LightGBM version: {lgb.__version__}")
except ImportError:
    print("Not installed — will need: !pip install lightgbm")

LightGBM version: 4.6.0


In [5]:
# shift every feature forward by 1 month, per ticker — so at date t,
# the features reflect what was knowable as of t-1, not t itself
data_sorted = data_sorted.sort_values(["ticker", "date"]).reset_index(drop=True)

for col in FEATURE_COLS:
    data_sorted[col] = data_sorted.groupby("ticker")[col].shift(1)

# drop rows that lost their features to the shift (each ticker's first obs)
data_sorted = data_sorted.dropna(subset=FEATURE_COLS + [TARGET_COL])

print(f"Rows after lag + dropna: {len(data_sorted):,}")

Rows after lag + dropna: 39,267


In [6]:
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

data_sorted = data.dropna(subset=FEATURE_COLS + [TARGET_COL]).sort_values("date").reset_index(drop=True)

fold_results = []

for fold_idx, f in enumerate(folds):
    train_mask = (data_sorted["date"] >= f["train_start"]) & (data_sorted["date"] <= f["train_end"])
    test_mask = (data_sorted["date"] >= f["test_start"]) & (data_sorted["date"] <= f["test_end"])

    train = data_sorted[train_mask]
    test = data_sorted[test_mask]

    if len(train) < 50 or len(test) < 5:
        continue

    X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
    X_test, y_test = test[FEATURE_COLS], test[TARGET_COL]

    model = lgb.LGBMRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        min_child_samples=20, random_state=42, verbose=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    test_copy = test.copy()
    test_copy["pred"] = preds
    monthly_ic = test_copy.groupby("date").apply(
        lambda g: spearmanr(g["pred"], g[TARGET_COL]).correlation if len(g) > 5 else np.nan
    )

    fold_results.append({
        "fold": fold_idx,
        "test_start": f["test_start"], "test_end": f["test_end"],
        "n_train": len(train), "n_test": len(test),
        "mean_ic": monthly_ic.mean(),
    })

results_df = pd.DataFrame(fold_results)
print(f"Folds completed: {len(results_df)}")
print(results_df.head(10))
print(f"\nOverall mean IC across all folds: {results_df['mean_ic'].mean():.4f}")
print(f"IC std: {results_df['mean_ic'].std():.4f}")

Folds completed: 54
   fold test_start   test_end  n_train  n_test   mean_ic
0     0 2013-03-31 2013-05-31     1240     159  0.573389
1     1 2013-06-30 2013-08-31     1396     165  0.467322
2     2 2013-09-30 2013-11-30     1558     179  0.684322
3     3 2013-12-31 2014-02-28     1734     198  0.475835
4     4 2014-03-31 2014-05-31     1826     235  0.371357
5     5 2014-06-30 2014-08-31     1930     263  0.613407
6     6 2014-09-30 2014-11-30     2065     236  0.825052
7     7 2014-12-31 2015-02-28     2158     218  0.856746
8     8 2015-03-31 2015-05-31     2229     217  0.882777
9     9 2015-06-30 2015-08-31     2309     216  0.783278

Overall mean IC across all folds: 0.7237
IC std: 0.1151


In [7]:
print(data_sorted[["ticker","date","mom_1m","xs_excess_ret"]].head(10))
print(f"\nRows after lag: {len(data_sorted):,}")

  ticker       date    mom_1m  xs_excess_ret
0   XMEM 2010-03-31  0.033067            0.0
1   XMEM 2010-04-30 -0.002948            0.0
2   XMEM 2010-05-31 -0.043608            0.0
3   XMEM 2010-06-30 -0.002185            0.0
4   XMEM 2010-07-31  0.069782            0.0
5   XMEM 2010-08-31  0.004465            0.0
6   XMEM 2010-09-30  0.055456            0.0
7   XMEM 2010-10-31  0.008786            0.0
8   XMEM 2010-11-30 -0.011290            0.0
9   XMEM 2010-12-31  0.071680            0.0

Rows after lag: 40,092


In [8]:
# rebuild from scratch — don't rely on any previously-run cell's state
data_sorted = data.dropna(subset=FEATURE_COLS + [TARGET_COL]).sort_values(["ticker","date"]).reset_index(drop=True)

# apply the lag — features at t must reflect info known before period t
for col in FEATURE_COLS:
    data_sorted[col] = data_sorted.groupby("ticker")[col].shift(1)
data_sorted = data_sorted.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

print(f"Rows after lag + dropna: {len(data_sorted):,}")
print(data_sorted[["ticker","date","mom_1m","xs_excess_ret"]].head(5))

# re-run the full walk-forward loop against the NOW-CORRECTLY-LAGGED data
import lightgbm as lgb
from scipy.stats import spearmanr
import numpy as np

fold_results = []
for fold_idx, f in enumerate(folds):
    train_mask = (data_sorted["date"] >= f["train_start"]) & (data_sorted["date"] <= f["train_end"])
    test_mask = (data_sorted["date"] >= f["test_start"]) & (data_sorted["date"] <= f["test_end"])
    train = data_sorted[train_mask]
    test = data_sorted[test_mask]
    if len(train) < 50 or len(test) < 5:
        continue
    X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
    X_test = test[FEATURE_COLS]
    model = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                               min_child_samples=20, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    test_copy = test.copy()
    test_copy["pred"] = model.predict(X_test)
    monthly_ic = test_copy.groupby("date").apply(
        lambda g: spearmanr(g["pred"], g[TARGET_COL]).correlation if len(g) > 5 else np.nan)
    fold_results.append({"fold": fold_idx, "test_start": f["test_start"], "test_end": f["test_end"],
                          "n_train": len(train), "n_test": len(test), "mean_ic": monthly_ic.mean()})

results_df = pd.DataFrame(fold_results)
print(f"\nFolds completed: {len(results_df)}")
print(results_df.head(10))
print(f"\nOverall mean IC across all folds: {results_df['mean_ic'].mean():.4f}")
print(f"IC std: {results_df['mean_ic'].std():.4f}")

Rows after lag + dropna: 39,267
  ticker       date    mom_1m  xs_excess_ret
0   100D 2022-05-31  0.004987       0.018486
1   100D 2022-11-30  0.009141       0.031728
2   100D 2022-12-31  0.060121      -0.002051
3   100D 2023-01-31 -0.021285      -0.014815
4   100D 2023-02-28  0.040429       0.024489

Folds completed: 54
   fold test_start   test_end  n_train  n_test   mean_ic
0     0 2013-03-31 2013-05-31     1153     157 -0.004276
1     1 2013-06-30 2013-08-31     1308     164 -0.020025
2     2 2013-09-30 2013-11-30     1469     172 -0.102751
3     3 2013-12-31 2014-02-28     1638     191  0.104981
4     4 2014-03-31 2014-05-31     1775     226  0.002718
5     5 2014-06-30 2014-08-31     1875     260  0.047440
6     6 2014-09-30 2014-11-30     2012     224 -0.115201
7     7 2014-12-31 2015-02-28     2096     218 -0.101688
8     8 2015-03-31 2015-05-31     2169     209  0.074862
9     9 2015-06-30 2015-08-31     2244     213 -0.028132

Overall mean IC across all folds: -0.0168
IC std:

In [9]:
from scipy import stats

ic_values = results_df["mean_ic"].dropna()
t_stat, p_value = stats.ttest_1samp(ic_values, 0)
print(f"Mean IC: {ic_values.mean():.4f}")
print(f"t-stat vs zero: {t_stat:.3f}")
print(f"p-value: {p_value:.3f}")

Mean IC: -0.0168
t-stat vs zero: -1.116
p-value: 0.269


In [11]:
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)

out_path = ROOT / "results/tables/v1_walkforward_ic_results.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

summary = {
    "version": "V1 (price-only features)",
    "features": FEATURE_COLS,
    "n_folds": len(results_df),
    "mean_ic": float(ic_values.mean()),
    "ic_std": float(ic_values.std()),
    "t_stat": float(t_stat),
    "p_value": float(p_value),
}
import json
with open(ROOT / "results/tables/v1_summary.json", "w") as fjson:
    json.dump(summary, fjson, indent=2)
print(summary)

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\results\tables\v1_walkforward_ic_results.csv
{'version': 'V1 (price-only features)', 'features': ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'realized_vol', 'max_drawdown_12m'], 'n_folds': 54, 'mean_ic': -0.016837644149653653, 'ic_std': 0.1108753272918088, 't_stat': -1.115946287905753, 'p_value': 0.26947926509508424}


In [12]:
!{sys.executable} -m pip install pandas-datareader --quiet

import pandas_datareader.data as web
import datetime

start = datetime.datetime(2008, 1, 1)
end = datetime.datetime(2026, 10, 31)

series_codes = {
    "uk_rate": "INTDSRGBM193N",
    "uk_gilt_10y": "IRLTLT01GBM156N",
    "gbpusd": "DEXUSUK",
    "vix": "VIXCLS",
}

macro_raw = {}
for name, code in series_codes.items():
    try:
        s = web.DataReader(code, "fred", start, end)
        macro_raw[name] = s
        print(f"{name} ({code}): {len(s)} rows, {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        print(f"{name} ({code}): FAILED — {e}")

uk_rate (INTDSRGBM193N): 63 rows, 2008-01-01 to 2013-03-01
uk_gilt_10y (IRLTLT01GBM156N): 224 rows, 2008-01-01 to 2026-08-01
gbpusd (DEXUSUK): 4894 rows, 2008-01-01 to 2026-10-02
vix (VIXCLS): 4897 rows, 2008-01-01 to 2026-10-07


In [13]:
series_codes = {
    "uk_rate": "IUDZLS9",       # SONIA (90th pctile) — live BoE short-rate proxy, replaces discontinued BOERUKM
    "uk_gilt_10y": "IRLTLT01GBM156N",
    "gbpusd": "DEXUSUK",
    "vix": "VIXCLS",
}

macro_raw = {}
for name, code in series_codes.items():
    try:
        s = web.DataReader(code, "fred", start, end)
        macro_raw[name] = s
        print(f"{name} ({code}): {len(s)} rows, {s.index.min().date()} to {s.index.max().date()}")
    except Exception as e:
        print(f"{name} ({code}): FAILED — {e}")

uk_rate (IUDZLS9): 2207 rows, 2018-04-23 to 2026-10-06
uk_gilt_10y (IRLTLT01GBM156N): 224 rows, 2008-01-01 to 2026-08-01
gbpusd (DEXUSUK): 4894 rows, 2008-01-01 to 2026-10-02
vix (VIXCLS): 4897 rows, 2008-01-01 to 2026-10-07


In [14]:
# resample each macro series to month-end, forward-fill small gaps (weekends/holidays)
macro_monthly = {}
for name, s in macro_raw.items():
    col = s.columns[0]
    monthly = s[col].resample("ME").last().ffill()
    macro_monthly[name] = monthly

macro_df = pd.DataFrame(macro_monthly).reset_index()
macro_df = macro_df.rename(columns={"DATE": "date"})

print(f"Macro monthly rows: {len(macro_df)}")
print(f"Date range: {macro_df['date'].min().date()} to {macro_df['date'].max().date()}")
print(f"\nNaN counts per column:\n{macro_df.isna().sum()}")
macro_df.head()

Macro monthly rows: 226
Date range: 2008-01-31 to 2026-10-31

NaN counts per column:
date             0
uk_rate        123
uk_gilt_10y      2
gbpusd           0
vix              0
dtype: int64


,date,uk_rate,uk_gilt_10y,gbpusd,vix
0,2008-01-31,NaN,4.4925,1.9895,26.20
1,2008-02-29,NaN,4.6234,1.9864,26.54
2,2008-03-31,NaN,4.4450,1.9855,25.61
3,2008-04-30,NaN,4.6432,1.9847,20.79
4,2008-05-31,NaN,4.8683,1.9793,17.83


In [15]:
# merge macro into the main dataset
data_v2 = data.merge(macro_df, on="date", how="left")

MACRO_COLS = ["uk_rate", "uk_gilt_10y", "gbpusd", "vix"]
FEATURE_COLS_V2 = FEATURE_COLS + MACRO_COLS

# apply the SAME lag discipline as before: shift macro by 1 month per the
# global date index (not per-ticker, since macro is shared across all tickers
# in a given month) — but to keep logic simple and consistent, lag macro the
# same way as price features, via merge-then-shift by date
macro_lagged = macro_df.copy()
macro_lagged[MACRO_COLS] = macro_lagged[MACRO_COLS].shift(1)  # lag by one calendar month

data_v2 = data.merge(macro_lagged, on="date", how="left")
data_v2 = data_v2.sort_values(["ticker", "date"]).reset_index(drop=True)

# re-apply the price-feature lag exactly as before (V1 logic)
for col in FEATURE_COLS:
    data_v2[col] = data_v2.groupby("ticker")[col].shift(1)

data_v2 = data_v2.dropna(subset=FEATURE_COLS_V2 + [TARGET_COL]).reset_index(drop=True)

print(f"Rows after full lag + dropna: {len(data_v2):,}")
print(f"Date range: {data_v2['date'].min().date()} to {data_v2['date'].max().date()}")
data_v2[["ticker","date"] + FEATURE_COLS_V2 + [TARGET_COL]].head()

Rows after full lag + dropna: 32,591
Date range: 2018-05-31 to 2026-09-30


,ticker,date,mom_1m,mom_3m,mom_6m,mom_12m,realized_vol,max_drawdown_12m,uk_rate,uk_gilt_10y,gbpusd,vix,xs_excess_ret
0,100D,2022-05-31,0.004987,0.021975,0.043493,0.083436,0.188796,-0.088355,0.72,1.8362,1.2565,33.40,0.018486
1,100D,2022-11-30,0.009141,0.029759,0.079605,0.069738,0.165207,-0.088355,2.20,4.1096,1.1515,25.88,0.031728
2,100D,2022-12-31,0.060121,0.068966,0.002221,0.062972,0.140565,-0.112612,2.95,3.4171,1.1962,20.58,-0.002051
3,100D,2023-01-31,-0.021285,0.070645,0.021211,0.004067,0.115686,-0.112612,3.46,3.5015,1.2077,21.67,-0.014815
4,100D,2023-02-28,0.040429,0.081217,0.044804,0.035274,0.112862,-0.112612,3.46,3.5115,1.2324,19.40,0.024489


In [16]:
# rebuild without dropping on uk_rate specifically — let LightGBM handle its NaNs natively
macro_lagged = macro_df.copy()
macro_lagged[MACRO_COLS] = macro_lagged[MACRO_COLS].shift(1)

data_v2 = data.merge(macro_lagged, on="date", how="left")
data_v2 = data_v2.sort_values(["ticker", "date"]).reset_index(drop=True)

for col in FEATURE_COLS:
    data_v2[col] = data_v2.groupby("ticker")[col].shift(1)

# drop only on price features + target + the macro cols that are reliably complete
# (uk_gilt_10y, gbpusd, vix) — leave uk_rate's NaNs in place for LightGBM to handle
required_complete = FEATURE_COLS + ["uk_gilt_10y", "gbpusd", "vix", TARGET_COL]
data_v2 = data_v2.dropna(subset=required_complete).reset_index(drop=True)

print(f"Rows after lag + partial dropna: {len(data_v2):,}")
print(f"Date range: {data_v2['date'].min().date()} to {data_v2['date'].max().date()}")
print(f"uk_rate NaN count remaining: {data_v2['uk_rate'].isna().sum():,} of {len(data_v2):,}")

Rows after lag + partial dropna: 38,709
Date range: 2010-04-30 to 2026-09-30
uk_rate NaN count remaining: 6,118 of 38,709


In [17]:
FEATURE_COLS_V2 = FEATURE_COLS + MACRO_COLS

fold_results_v2 = []
for fold_idx, f in enumerate(folds):
    train_mask = (data_v2["date"] >= f["train_start"]) & (data_v2["date"] <= f["train_end"])
    test_mask = (data_v2["date"] >= f["test_start"]) & (data_v2["date"] <= f["test_end"])
    train = data_v2[train_mask]
    test = data_v2[test_mask]
    if len(train) < 50 or len(test) < 5:
        continue
    X_train, y_train = train[FEATURE_COLS_V2], train[TARGET_COL]
    X_test = test[FEATURE_COLS_V2]
    model = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                               min_child_samples=20, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    test_copy = test.copy()
    test_copy["pred"] = model.predict(X_test)
    monthly_ic = test_copy.groupby("date").apply(
        lambda g: spearmanr(g["pred"], g[TARGET_COL]).correlation if len(g) > 5 else np.nan)
    fold_results_v2.append({"fold": fold_idx, "test_start": f["test_start"], "test_end": f["test_end"],
                             "n_train": len(train), "n_test": len(test), "mean_ic": monthly_ic.mean()})

results_df_v2 = pd.DataFrame(fold_results_v2)
ic_values_v2 = results_df_v2["mean_ic"].dropna()
t_stat_v2, p_value_v2 = stats.ttest_1samp(ic_values_v2, 0)

print(f"Folds completed: {len(results_df_v2)}")
print(f"\nV1 (price-only):  mean IC = {ic_values.mean():.4f}, t = {t_stat:.3f}, p = {p_value:.3f}")
print(f"V2 (+ macro):      mean IC = {ic_values_v2.mean():.4f}, t = {t_stat_v2:.3f}, p = {p_value_v2:.3f}")

Folds completed: 54

V1 (price-only):  mean IC = -0.0168, t = -1.116, p = 0.269
V2 (+ macro):      mean IC = 0.0031, t = 0.168, p = 0.867


In [18]:
# feature importance from the LAST fold's model (most recent, most data)
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS_V2,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df)

            feature  importance
0            mom_1m         384
1            mom_3m         286
9               vix         273
3           mom_12m         250
7       uk_gilt_10y         248
4      realized_vol         245
5  max_drawdown_12m         234
8            gbpusd         226
2            mom_6m         225
6           uk_rate         137


In [19]:
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
results_df_v2.to_csv(ROOT / "results/tables/v2_walkforward_ic_results.csv", index=False)

summary_v2 = {
    "version": "V2 (price + macro features)",
    "features": FEATURE_COLS_V2,
    "n_folds": len(results_df_v2),
    "mean_ic": float(ic_values_v2.mean()),
    "ic_std": float(ic_values_v2.std()),
    "t_stat": float(t_stat_v2),
    "p_value": float(p_value_v2),
    "feature_importance_last_fold": importance_df.set_index("feature")["importance"].to_dict(),
    "conclusion": "Macro features receive substantial model importance (vix, uk_gilt_10y, gbpusd comparable to price features) but do not produce statistically significant improvement in monthly cross-sectional rank IC vs V1."
}
import json
with open(ROOT / "results/tables/v2_summary.json", "w") as fjson:
    json.dump(summary_v2, fjson, indent=2)

print("V2 results saved.")
print(f"V1 mean IC: {ic_values.mean():.4f} (p={p_value:.3f})")
print(f"V2 mean IC: {ic_values_v2.mean():.4f} (p={p_value_v2:.3f})")

V2 results saved.
V1 mean IC: -0.0168 (p=0.269)
V2 mean IC: 0.0031 (p=0.867)


In [20]:
# rebuild from the clean, unlagged source to avoid compounding dropna issues
base = pd.read_csv(ROOT / "data/processed/uk_etf_features_targets.csv", parse_dates=["date"])
base = base.sort_values(["ticker", "date"]).reset_index(drop=True)

# 3-month forward cumulative return, per ticker
base["fwd_ret_3m"] = (
    base.groupby("ticker")["ret"]
    .transform(lambda s: (1 + s.shift(-1)) * (1 + s.shift(-2)) * (1 + s.shift(-3)) - 1)
)

# cross-sectional demean, per month, among rows with a valid 3m forward target
base["xs_excess_ret_3m"] = base["fwd_ret_3m"] - base.groupby("date")["fwd_ret_3m"].transform("mean")

# apply point-in-time lag to price features (same as V1/V2)
for col in FEATURE_COLS:
    base[col] = base.groupby("ticker")[col].shift(1)

# merge macro (already lagged in macro_lagged from before)
data_v3 = base.merge(macro_lagged, on="date", how="left")

required_complete_v3 = FEATURE_COLS + ["uk_gilt_10y","gbpusd","vix","xs_excess_ret_3m"]
data_v3 = data_v3.dropna(subset=required_complete_v3).reset_index(drop=True)

print(f"Rows: {len(data_v3):,}")
print(f"Date range: {data_v3['date'].min().date()} to {data_v3['date'].max().date()}")
data_v3[["ticker","date","fwd_ret_3m","xs_excess_ret_3m"]].head()

Rows: 36,930
Date range: 2010-04-30 to 2026-07-31


,ticker,date,fwd_ret_3m,xs_excess_ret_3m
0,100D,2022-05-31,0.081217,0.087703
1,100D,2022-11-30,0.025921,0.007397
2,100D,2022-12-31,0.032290,-0.015578
3,100D,2023-01-31,0.055446,0.061654
4,100D,2023-02-28,0.056622,0.064603


In [21]:
TARGET_COL_V3 = "xs_excess_ret_3m"

fold_results_v3 = []
for fold_idx, f in enumerate(folds):
    train_mask = (data_v3["date"] >= f["train_start"]) & (data_v3["date"] <= f["train_end"])
    test_mask = (data_v3["date"] >= f["test_start"]) & (data_v3["date"] <= f["test_end"])
    train = data_v3[train_mask]
    test = data_v3[test_mask]
    if len(train) < 50 or len(test) < 5:
        continue
    X_train, y_train = train[FEATURE_COLS_V2], train[TARGET_COL_V3]
    X_test = test[FEATURE_COLS_V2]
    model = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                               min_child_samples=20, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    test_copy = test.copy()
    test_copy["pred"] = model.predict(X_test)
    monthly_ic = test_copy.groupby("date").apply(
        lambda g: spearmanr(g["pred"], g[TARGET_COL_V3]).correlation if len(g) > 5 else np.nan)
    fold_results_v3.append({"fold": fold_idx, "test_start": f["test_start"], "test_end": f["test_end"],
                             "n_train": len(train), "n_test": len(test), "mean_ic": monthly_ic.mean()})

results_df_v3 = pd.DataFrame(fold_results_v3)
ic_values_v3 = results_df_v3["mean_ic"].dropna()
t_stat_v3, p_value_v3 = stats.ttest_1samp(ic_values_v3, 0)

print(f"Folds completed: {len(results_df_v3)}")
print(f"\nV1 (1m, price-only):  mean IC = {ic_values.mean():.4f}, t = {t_stat:.3f}, p = {p_value:.3f}")
print(f"V2 (1m, +macro):       mean IC = {ic_values_v2.mean():.4f}, t = {t_stat_v2:.3f}, p = {p_value_v2:.3f}")
print(f"V3 (3m, +macro):       mean IC = {ic_values_v3.mean():.4f}, t = {t_stat_v3:.3f}, p = {p_value_v3:.3f}")

Folds completed: 54

V1 (1m, price-only):  mean IC = -0.0168, t = -1.116, p = 0.269
V2 (1m, +macro):       mean IC = 0.0031, t = 0.168, p = 0.867
V3 (3m, +macro):       mean IC = 0.0377, t = 1.661, p = 0.103


In [22]:
# 1. Newey-West adjusted t-stat (accounts for serial correlation in overlapping windows)
import statsmodels.api as sm

X_const = sm.add_constant(np.ones(len(ic_values_v3)))
nw_model = sm.OLS(ic_values_v3.values, X_const).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
print("Newey-West adjusted (lag=3, matching the 3-month overlap):")
print(f"Mean IC: {nw_model.params[0]:.4f}")
print(f"t-stat: {nw_model.tvalues[0]:.3f}")
print(f"p-value: {nw_model.pvalues[0]:.3f}")

# 2. fold-by-fold stability check
print(f"\nFolds with positive IC: {(ic_values_v3 > 0).sum()} of {len(ic_values_v3)} ({(ic_values_v3>0).mean()*100:.0f}%)")
print(f"\nFold-by-fold IC (chronological):")
print(results_df_v3[["test_start","mean_ic"]].to_string(index=False))

Newey-West adjusted (lag=3, matching the 3-month overlap):
Mean IC: 0.0377
t-stat: 2.404
p-value: 0.016

Folds with positive IC: 36 of 54 (67%)

Fold-by-fold IC (chronological):
test_start   mean_ic
2013-03-31  0.271561
2013-06-30 -0.064162
2013-09-30  0.267672
2013-12-31 -0.040819
2014-03-31 -0.213240
2014-06-30  0.211696
2014-09-30  0.173740
2014-12-31 -0.027004
2015-03-31  0.116821
2015-06-30  0.253599
2015-09-30 -0.010525
2015-12-31 -0.256587
2016-03-31  0.081341
2016-06-30  0.332219
2016-09-30  0.059064
2016-12-31  0.137682
2017-03-31  0.012165
2017-06-30 -0.036346
2017-09-30  0.257649
2017-12-31  0.034164
2018-03-31 -0.138813
2018-06-30  0.050176
2018-09-30 -0.176163
2018-12-31  0.001768
2019-03-31  0.003679
2019-06-30 -0.074329
2019-09-30  0.196525
2019-12-31  0.038612
2020-03-31 -0.107580
2020-06-30  0.172537
2020-09-30 -0.177979
2020-12-31  0.069582
2021-03-31  0.092783
2021-06-30  0.102620
2021-09-30  0.088758
2021-12-31  0.079123
2022-03-31  0.116389
2022-06-30  0.108430
202

In [23]:
(ROOT / "results/tables").mkdir(parents=True, exist_ok=True)
results_df_v3.to_csv(ROOT / "results/tables/v3_walkforward_ic_results.csv", index=False)

summary_v3 = {
    "version": "V3 (3-month horizon, price + macro features, LightGBM)",
    "features": FEATURE_COLS_V2,
    "n_folds": len(results_df_v3),
    "mean_ic": float(ic_values_v3.mean()),
    "newey_west_t_stat": float(nw_model.tvalues[0]),
    "newey_west_p_value": float(nw_model.pvalues[0]),
    "pct_folds_positive": float((ic_values_v3 > 0).mean()),
    "conclusion": "Statistically significant (NW p=0.016) cross-sectional ranking signal at 3-month horizon, not present at 1-month. Two large negative outlier folds (2022-09, 2023-09) coincide with rate-hiking/bond-stress periods — flagged for regime analysis."
}
import json
with open(ROOT / "results/tables/v3_summary.json", "w") as fjson:
    json.dump(summary_v3, fjson, indent=2)
print("V3 saved.")

V3 saved.


In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

models_to_test = {
    "LR": LinearRegression(),
    "RF": RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1),
}

model_comparison = {"LightGBM": ic_values_v3}

for model_name, model_obj in models_to_test.items():
    fold_ics = []
    for f in folds:
        train_mask = (data_v3["date"] >= f["train_start"]) & (data_v3["date"] <= f["train_end"])
        test_mask = (data_v3["date"] >= f["test_start"]) & (data_v3["date"] <= f["test_end"])
        train, test = data_v3[train_mask], data_v3[test_mask]
        if len(train) < 50 or len(test) < 5:
            continue
        X_train, y_train = train[FEATURE_COLS_V2].fillna(0), train[TARGET_COL_V3]
        X_test = test[FEATURE_COLS_V2].fillna(0)
        model_obj.fit(X_train, y_train)
        preds = model_obj.predict(X_test)
        test_copy = test.copy()
        test_copy["pred"] = preds
        monthly_ic = test_copy.groupby("date").apply(
            lambda g: spearmanr(g["pred"], g[TARGET_COL_V3]).correlation if len(g) > 5 else np.nan)
        fold_ics.append(monthly_ic.mean())
    model_comparison[model_name] = pd.Series(fold_ics).dropna()
    print(f"{model_name}: mean IC = {model_comparison[model_name].mean():.4f}")

print("\n--- Model comparison summary ---")
for name, ics in model_comparison.items():
    t, p = stats.ttest_1samp(ics, 0)
    print(f"{name}: mean IC={ics.mean():.4f}, naive t={t:.2f}, naive p={p:.3f}")
    

LR: mean IC = -0.0468
RF: mean IC = 0.0425

--- Model comparison summary ---
LightGBM: mean IC=0.0377, naive t=1.66, naive p=0.103
LR: mean IC=-0.0468, naive t=-1.56, naive p=0.126
RF: mean IC=0.0425, naive t=1.76, naive p=0.084


In [25]:
nw_results = {}
for name, ics in model_comparison.items():
    X_const = sm.add_constant(np.ones(len(ics)))
    nw_fit = sm.OLS(ics.values, X_const).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
    nw_results[name] = {"mean_ic": ics.mean(), "nw_t": nw_fit.tvalues[0], "nw_p": nw_fit.pvalues[0]}
    print(f"{name}: mean IC={ics.mean():.4f}, NW t={nw_fit.tvalues[0]:.3f}, NW p={nw_fit.pvalues[0]:.4f}")

LightGBM: mean IC=0.0377, NW t=2.404, NW p=0.0162
LR: mean IC=-0.0468, NW t=-2.021, NW p=0.0433
RF: mean IC=0.0425, NW t=2.562, NW p=0.0104


In [26]:
comparison_summary = pd.DataFrame(nw_results).T
comparison_summary.index.name = "model"
comparison_summary.to_csv(ROOT / "results/tables/v3_model_comparison.csv")
print(comparison_summary)

# consolidated notebook 04 summary — all versions, for easy reference going forward
notebook04_summary = {
    "V1_price_only_1m": {"mean_ic": float(ic_values.mean()), "p_value": float(p_value)},
    "V2_price_macro_1m": {"mean_ic": float(ic_values_v2.mean()), "p_value": float(p_value_v2)},
    "V3_price_macro_3m_LightGBM": {"mean_ic": float(ic_values_v3.mean()), "nw_p_value": float(nw_results["LightGBM"]["nw_p"])},
    "V3_price_macro_3m_LR": {"mean_ic": float(nw_results["LR"]["mean_ic"]), "nw_p_value": float(nw_results["LR"]["nw_p"])},
    "V3_price_macro_3m_RF": {"mean_ic": float(nw_results["RF"]["mean_ic"]), "nw_p_value": float(nw_results["RF"]["nw_p"])},
    "best_model": "Random Forest, 3-month horizon, price+macro features",
    "key_finding": "No signal detected at 1-month horizon (price-only or +macro). At 3-month horizon, tree-based models (RF, LightGBM) show statistically significant positive rank IC after Newey-West adjustment for serial correlation; Linear Regression shows significant NEGATIVE IC, indicating the signal is nonlinear.",
}

with open(ROOT / "results/tables/notebook04_summary.json", "w") as fjson:
    json.dump(notebook04_summary, fjson, indent=2)

print("\nNotebook 04 complete. All results saved to results/tables/")

           mean_ic      nw_t      nw_p
model                                 
LightGBM  0.037691  2.404123  0.016211
LR       -0.046800 -2.021054  0.043274
RF        0.042469  2.561880  0.010411

Notebook 04 complete. All results saved to results/tables/


In [27]:
def run_ensemble(model_names, data_v3, folds, feature_cols, target_col):
    fold_ics = []
    for f in folds:
        train_mask = (data_v3["date"] >= f["train_start"]) & (data_v3["date"] <= f["train_end"])
        test_mask = (data_v3["date"] >= f["test_start"]) & (data_v3["date"] <= f["test_end"])
        train, test = data_v3[train_mask], data_v3[test_mask]
        if len(train) < 50 or len(test) < 5:
            continue

        X_train_raw, y_train = train[feature_cols], train[target_col]
        X_test_raw = test[feature_cols]
        X_train_filled, X_test_filled = X_train_raw.fillna(0), X_test_raw.fillna(0)

        preds = {}
        if "LightGBM" in model_names:
            m = lgb.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                   min_child_samples=20, random_state=42, verbose=-1)
            m.fit(X_train_raw, y_train)
            preds["LightGBM"] = m.predict(X_test_raw)
        if "RF" in model_names:
            m = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20,
                                       random_state=42, n_jobs=-1)
            m.fit(X_train_filled, y_train)
            preds["RF"] = m.predict(X_test_filled)
        if "LR" in model_names:
            m = LinearRegression()
            m.fit(X_train_filled, y_train)
            preds["LR"] = m.predict(X_test_filled)

        # rank-average ensemble: average each model's within-fold percentile rank
        # (more robust than averaging raw predictions across models with different scales)
        pred_df = pd.DataFrame(preds)
        ranked = pred_df.rank(pct=True)
        blended_pred = ranked.mean(axis=1)

        test_copy = test.copy()
        test_copy["pred"] = blended_pred.values
        monthly_ic = test_copy.groupby("date").apply(
            lambda g: spearmanr(g["pred"], g[target_col]).correlation if len(g) > 5 else np.nan)
        fold_ics.append(monthly_ic.mean())
    return pd.Series(fold_ics).dropna()

ensemble_rf_lgbm = run_ensemble(["RF","LightGBM"], data_v3, folds, FEATURE_COLS_V2, TARGET_COL_V3)
ensemble_all3 = run_ensemble(["RF","LightGBM","LR"], data_v3, folds, FEATURE_COLS_V2, TARGET_COL_V3)

for name, ics in [("Ensemble (RF+LightGBM)", ensemble_rf_lgbm), ("Ensemble (all 3)", ensemble_all3)]:
    X_const = sm.add_constant(np.ones(len(ics)))
    nw_fit = sm.OLS(ics.values, X_const).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
    print(f"{name}: mean IC={ics.mean():.4f}, NW t={nw_fit.tvalues[0]:.3f}, NW p={nw_fit.pvalues[0]:.4f}")

Ensemble (RF+LightGBM): mean IC=0.0438, NW t=2.788, NW p=0.0053
Ensemble (all 3): mean IC=0.0187, NW t=0.999, NW p=0.3178


In [28]:
final_comparison = pd.DataFrame({
    "LightGBM": {"mean_ic": ic_values_v3.mean(), "nw_p": nw_results["LightGBM"]["nw_p"]},
    "RF": {"mean_ic": nw_results["RF"]["mean_ic"], "nw_p": nw_results["RF"]["nw_p"]},
    "LR": {"mean_ic": nw_results["LR"]["mean_ic"], "nw_p": nw_results["LR"]["nw_p"]},
    "Ensemble_RF_LightGBM": {"mean_ic": ensemble_rf_lgbm.mean(),
        "nw_p": sm.OLS(ensemble_rf_lgbm.values, sm.add_constant(np.ones(len(ensemble_rf_lgbm)))).fit(cov_type='HAC', cov_kwds={'maxlags':3}).pvalues[0]},
    "Ensemble_all3": {"mean_ic": ensemble_all3.mean(),
        "nw_p": sm.OLS(ensemble_all3.values, sm.add_constant(np.ones(len(ensemble_all3)))).fit(cov_type='HAC', cov_kwds={'maxlags':3}).pvalues[0]},
}).T
final_comparison.index.name = "model"
final_comparison.to_csv(ROOT / "results/tables/v3_full_model_comparison.csv")
print(final_comparison)

notebook04_summary["V3_ensemble_RF_LightGBM"] = {"mean_ic": float(ensemble_rf_lgbm.mean()), "nw_p_value": float(final_comparison.loc["Ensemble_RF_LightGBM","nw_p"])}
notebook04_summary["V3_ensemble_all3"] = {"mean_ic": float(ensemble_all3.mean()), "nw_p_value": float(final_comparison.loc["Ensemble_all3","nw_p"])}
notebook04_summary["best_model"] = "Ensemble (RF + LightGBM, rank-averaged), 3-month horizon, price+macro features"
notebook04_summary["key_finding"] += " A rank-averaged ensemble of RF+LightGBM outperforms either model alone (IC=0.044, NW p=0.005), while including LR in the ensemble degrades it (IC=0.019, p=0.32), confirming LR's negative signal is destructive rather than diversifying."

with open(ROOT / "results/tables/notebook04_summary.json", "w") as fjson:
    json.dump(notebook04_summary, fjson, indent=2)

print("\nNotebook 04 fully complete.")
print(f"Best model: Ensemble (RF+LightGBM) — mean IC {ensemble_rf_lgbm.mean():.4f}, NW p={final_comparison.loc['Ensemble_RF_LightGBM','nw_p']:.4f}")

                       mean_ic      nw_p
model                                   
LightGBM              0.037691  0.016211
RF                    0.042469  0.010411
LR                   -0.046800  0.043274
Ensemble_RF_LightGBM  0.043768  0.005311
Ensemble_all3         0.018691  0.317770

Notebook 04 fully complete.
Best model: Ensemble (RF+LightGBM) — mean IC 0.0438, NW p=0.0053
